# 05 — Spark Aggregations (the serving layer)

**Airline Operations Intelligence Platform** · Notebook 5 of 10 · *runs locally*

## Purpose
Implement Modules 5–8 of the project plan — flight operations, airline performance,
airport intelligence and route intelligence — and write **dashboard-ready aggregates**.

Each output matches a MongoDB collection schema from §17 of the proposal exactly, so
notebook `08` can push them without reshaping.

| Mart | Collection | Grain |
|---|---|---|
| `overall_kpis` | `overall_kpis` | one document |
| `airline_metrics` | `airline_metrics` | one per airline |
| `airport_metrics` | `airport_metrics` | one per airport |
| `route_metrics` | `route_metrics` | one per origin→destination |
| `time_trends` | `time_trends` | monthly / daily / hourly |
| `delay_distribution` | `delay_distribution` | one per delay bucket |
| `delay_causes` | `delay_causes` | one per cause |

## The architectural point (Unit 3)
All heavy computation happens **here**, in Spark, once. The dashboard never aggregates
5.8M rows — it reads a few hundred pre-computed documents. This is the
*precomputed serving layer* pattern, and it is why the dashboard can be fast on a laptop.

## Fair-comparison rules applied throughout
- **Cancelled and diverted flights are excluded from delay averages** — they have no
  arrival delay by definition (notebook 01, rules 2 and 3).
- **Cancellation rate is computed over all flights**, including cancelled ones.
- **Minimum-sample thresholds** are applied before any ranking (notebook 04, §4).

In [ ]:
import sys, time
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = build_spark("05-aggregations")

flights = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))
flights.cache()
N = flights.count()

# Reused predicates -- defined once so every mart uses the same definitions.
COMPLETED = F.col("status") == "completed"
CANCELLED = F.col("status") == "cancelled"
DIVERTED  = F.col("status") == "diverted"

MIN_AIRPORT_FLIGHTS = 10_000   # thresholds for ranking, per notebook 04
MIN_ROUTE_FLIGHTS   = 1_000

print(f"Curated flights : {N:,}")
print(f"Completed       : {flights.filter(COMPLETED).count():,}")
print(f"Marts output    : {PATHS['marts']}")

---
## 1. Overall KPIs

The headline numbers for the dashboard's executive overview.

In [ ]:
kpis = flights.agg(
    F.count("*").alias("total_flights"),
    F.sum(F.when(COMPLETED, 1).otherwise(0)).alias("completed_flights"),
    F.round(100.0 * F.avg(F.when(COMPLETED, 1 - F.col("is_delayed"))), 2).alias("on_time_pct"),
    F.round(F.avg(F.when(COMPLETED, F.col("dep_delay"))), 2).alias("avg_dep_delay"),
    F.round(F.avg(F.when(COMPLETED, F.col("arr_delay"))), 2).alias("avg_arr_delay"),
    F.round(100.0 * F.avg(F.when(CANCELLED, 1.0).otherwise(0.0)), 2).alias("cancellation_rate"),
    F.round(100.0 * F.avg(F.when(DIVERTED,  1.0).otherwise(0.0)), 2).alias("diversion_rate"),
    F.round(100.0 * F.avg(F.when(COMPLETED, F.col("is_delayed"))), 2).alias("delay_rate"),
    F.countDistinct("airline_code").alias("airlines"),
    F.countDistinct("origin").alias("airports"),
    F.countDistinct("route").alias("routes"),
)

kpis.show(vertical=True, truncate=False)

---
## 2. Airline performance (Module 6)

Ranked on delay rate, with the sample size carried alongside so no comparison is made
without its context.

`median_delay` uses `percentile_approx` — an exact median needs a full sort, which is a
wide operation on 5.8M rows; the approximate variant is computed within the aggregation.

In [ ]:
airline_metrics = (flights
    .groupBy("airline_code", "airline_name")
    .agg(
        F.count("*").alias("total_flights"),
        F.sum(F.when(COMPLETED, 1).otherwise(0)).alias("completed_flights"),
        F.round(100.0 * F.avg(F.when(COMPLETED, 1 - F.col("is_delayed"))), 2).alias("on_time_pct"),
        F.round(F.avg(F.when(COMPLETED, F.col("dep_delay"))), 2).alias("avg_dep_delay"),
        F.round(F.avg(F.when(COMPLETED, F.col("arr_delay"))), 2).alias("avg_arr_delay"),
        F.round(F.percentile_approx(F.when(COMPLETED, F.col("arr_delay")), 0.5), 2).alias("median_delay"),
        F.round(100.0 * F.avg(F.when(CANCELLED, 1.0).otherwise(0.0)), 2).alias("cancellation_rate"),
        F.round(100.0 * F.avg(F.when(DIVERTED,  1.0).otherwise(0.0)), 2).alias("diversion_rate"),
        F.round(100.0 * F.avg(F.when(COMPLETED, F.col("is_delayed"))), 2).alias("delay_rate"),
        F.round(F.avg("distance"), 0).alias("avg_distance"),
        F.countDistinct("route").alias("routes_served"),
    )
    .withColumn("rank_by_delay_rate", F.rank().over(Window.orderBy("delay_rate")))
    .orderBy("delay_rate"))

airline_metrics.select("airline_code","airline_name","total_flights","on_time_pct",
                       "avg_arr_delay","median_delay","delay_rate","cancellation_rate",
                       "rank_by_delay_rate").show(20, truncate=False)

In [ ]:
# Sanity: airline totals must sum to the dataset.
tot = airline_metrics.agg(F.sum("total_flights")).first()[0]
assert tot == N, f"airline totals {tot:,} != {N:,}"
print(f"Airline flight totals reconcile to the dataset: {tot:,}")

---
## 3. Airport intelligence (Module 7)

Airports are measured on **departures**, since departure delay is the airport's own
operational signal. `peak_delay_hour` is the hour of day with that airport's highest
delay rate — computed with a window function over hourly aggregates.

In [ ]:
airport_base = (flights
    .groupBy(F.col("origin").alias("airport_code"))
    .agg(
        F.first("origin_name").alias("airport_name"),
        F.first("origin_city").alias("city"),
        F.first("origin_state").alias("state"),
        F.first("origin_lat").alias("lat"),
        F.first("origin_lon").alias("lon"),
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.when(COMPLETED, F.col("dep_delay"))), 2).alias("avg_dep_delay"),
        F.round(F.avg(F.when(COMPLETED, F.col("arr_delay"))), 2).alias("avg_delay"),
        F.round(100.0 * F.avg(F.when(COMPLETED, F.col("is_delayed"))), 2).alias("delay_rate"),
        F.round(100.0 * F.avg(F.when(CANCELLED, 1.0).otherwise(0.0)), 2).alias("cancellation_rate"),
        F.countDistinct("airline_code").alias("airlines_served"),
        F.countDistinct("destination").alias("destinations_served"),
    ))

print(f"Airports: {airport_base.count()}")

In [ ]:
# Peak delay hour per airport, and peak-hour congestion ratio (for clustering in 07).
hourly = (flights.filter(COMPLETED)
    .groupBy(F.col("origin").alias("airport_code"), "sched_dep_hour")
    .agg(F.count("*").alias("hour_flights"),
         F.avg("is_delayed").alias("hour_delay_rate")))

w_delay = Window.partitionBy("airport_code").orderBy(F.desc("hour_delay_rate"))
w_vol   = Window.partitionBy("airport_code")

peak = (hourly
    .filter(F.col("hour_flights") >= 100)            # ignore near-empty hours
    .withColumn("rn", F.row_number().over(w_delay))
    .withColumn("busiest_hour_flights", F.max("hour_flights").over(w_vol))
    .withColumn("total_hour_flights",   F.sum("hour_flights").over(w_vol))
    .filter(F.col("rn") == 1)
    .select("airport_code",
            F.col("sched_dep_hour").alias("peak_delay_hour"),
            F.round(F.col("busiest_hour_flights") / F.col("total_hour_flights"), 4)
             .alias("peak_hour_congestion_ratio")))

airport_metrics = (airport_base.join(peak, "airport_code", "left")
    .withColumn("meets_min_sample", F.col("total_flights") >= MIN_AIRPORT_FLIGHTS))

(airport_metrics.filter("meets_min_sample")
    .orderBy(F.desc("delay_rate"))
    .select("airport_code","airport_name","total_flights","avg_delay","delay_rate",
            "cancellation_rate","peak_delay_hour")
    .show(10, truncate=False))

In [ ]:
n_ranked = airport_metrics.filter("meets_min_sample").count()
n_total  = airport_metrics.count()
print(f"Airports meeting the {MIN_AIRPORT_FLIGHTS:,}-flight ranking threshold: "
      f"{n_ranked} of {n_total}")
print("The rest are retained in the mart but flagged, so the dashboard can exclude")
print("them from rankings without discarding their data.")

---
## 4. Route intelligence (Module 8)

Origin→destination pairs, ranked for reliability. `reliability_rank` is 1 for the most
reliable route meeting the sample threshold.

In [ ]:
route_base = (flights
    .groupBy("route", "origin", "destination")
    .agg(
        F.first("origin_city").alias("origin_city"),
        F.first("dest_city").alias("dest_city"),
        F.count("*").alias("total_flights"),
        F.round(F.avg(F.when(COMPLETED, F.col("arr_delay"))), 2).alias("avg_delay"),
        F.round(F.percentile_approx(F.when(COMPLETED, F.col("arr_delay")), 0.5), 2).alias("median_delay"),
        F.round(100.0 * F.avg(F.when(COMPLETED, F.col("is_delayed"))), 2).alias("delay_rate"),
        F.round(100.0 * F.avg(F.when(CANCELLED, 1.0).otherwise(0.0)), 2).alias("cancellation_rate"),
        F.round(F.avg("distance"), 0).alias("distance"),
        F.countDistinct("airline_code").alias("airlines_serving"),
    )
    .withColumn("meets_min_sample", F.col("total_flights") >= MIN_ROUTE_FLIGHTS))

route_metrics = route_base.withColumn(
    "reliability_rank",
    F.when(F.col("meets_min_sample"),
           F.rank().over(Window.partitionBy("meets_min_sample").orderBy("delay_rate"))))

print(f"Routes total: {route_metrics.count():,}   "
      f"meeting {MIN_ROUTE_FLIGHTS:,}-flight threshold: "
      f"{route_metrics.filter('meets_min_sample').count():,}")

In [ ]:
print("MOST RELIABLE routes (>= 1,000 flights):")
(route_metrics.filter("meets_min_sample").orderBy("reliability_rank")
    .select("route","total_flights","avg_delay","delay_rate","reliability_rank")
    .show(10, truncate=False))

print("LEAST RELIABLE routes (>= 1,000 flights):")
(route_metrics.filter("meets_min_sample").orderBy(F.desc("reliability_rank"))
    .select("route","total_flights","avg_delay","delay_rate","reliability_rank")
    .show(10, truncate=False))

---
## 5. Time trends

Three grains in one collection, distinguished by a `dimension` field — matching the
`time_trends` schema in §17 of the proposal.

In [ ]:
def trend(group_cols, dimension, period_expr):
    return (flights.groupBy(*group_cols)
        .agg(F.count("*").alias("total_flights"),
             F.round(F.avg(F.when(COMPLETED, F.col("arr_delay"))), 2).alias("avg_delay"),
             F.round(F.avg(F.when(COMPLETED, F.col("dep_delay"))), 2).alias("avg_dep_delay"),
             F.round(100.0 * F.avg(F.when(COMPLETED, F.col("is_delayed"))), 2).alias("delay_rate"),
             F.round(100.0 * F.avg(F.when(CANCELLED, 1.0).otherwise(0.0)), 2).alias("cancellation_rate"))
        .select(F.lit(dimension).alias("dimension"),
                period_expr.alias("period"),
                "total_flights", "avg_delay", "avg_dep_delay",
                "delay_rate", "cancellation_rate"))

monthly = trend(["month"], "monthly",
                F.concat(F.lit("2015-"), F.lpad(F.col("month").cast("string"), 2, "0")))
dow     = trend(["day_of_week"], "day_of_week", F.col("day_of_week").cast("string"))
hourly_t= trend(["sched_dep_hour"], "hourly", F.col("sched_dep_hour").cast("string"))
seasonal= trend(["season"], "seasonal", F.col("season"))

time_trends = monthly.unionByName(dow).unionByName(hourly_t).unionByName(seasonal)

monthly.orderBy("period").show(12, truncate=False)

In [ ]:
print("Delay rate by day of week (1=Mon ... 7=Sun):")
dow.orderBy("period").show(truncate=False)
print("Delay rate by scheduled departure hour:")
hourly_t.orderBy(F.col("period").cast("int")).show(24, truncate=False)

---
## 6. Delay distribution and causes

The buckets come from `delay_category`, derived in notebook 02. Percentages are over
**completed** flights only, since cancelled flights have no arrival delay.

In [ ]:
n_completed = flights.filter(COMPLETED).count()

delay_distribution = (flights.filter(COMPLETED)
    .groupBy(F.col("delay_category").alias("delay_bucket"))
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(100.0 * F.col("count") / F.lit(n_completed), 2))
    .orderBy(F.desc("count")))

delay_distribution.show(truncate=False)

In [ ]:
# Delay causes: only defined for flights arriving 15+ minutes late (rule 3).
late = flights.filter(COMPLETED & (F.col("arr_delay") >= 15))
n_late = late.count()

causes = [("carrier","delay_carrier"), ("weather","delay_weather"),
          ("nas","delay_nas"), ("security","delay_security"),
          ("late_aircraft","delay_late_aircraft")]

rows = late.agg(*[F.sum(col).alias(name) for name, col in causes]).first().asDict()
total_min = sum(rows.values())

delay_causes = spark.createDataFrame(
    [(name, int(rows[name]), round(100.0*rows[name]/total_min, 2),
      round(rows[name]/n_late, 2)) for name, _ in causes],
    "cause string, total_minutes long, pct_of_delay_minutes double, avg_minutes_per_late_flight double"
).orderBy(F.desc("total_minutes"))

print(f"Flights arriving 15+ min late: {n_late:,}")
delay_causes.show(truncate=False)

---
## 7. Airline × airport, for the dashboard's drill-down

Supports "how does airline X perform at airport Y" without the dashboard joining anything.

In [ ]:
airline_airport = (flights
    .filter(F.col("origin").isNotNull())
    .groupBy("airline_code", "airline_name", F.col("origin").alias("airport_code"))
    .agg(F.count("*").alias("total_flights"),
         F.round(F.avg(F.when(COMPLETED, F.col("arr_delay"))), 2).alias("avg_delay"),
         F.round(100.0 * F.avg(F.when(COMPLETED, F.col("is_delayed"))), 2).alias("delay_rate"))
    .filter(F.col("total_flights") >= 500))

print(f"Airline x airport pairs (>= 500 flights): {airline_airport.count():,}")
airline_airport.orderBy(F.desc("total_flights")).show(5, truncate=False)

---
## 8. Write the marts

Each mart is small — hundreds to a few thousand rows — so they are written **unpartitioned
and coalesced to a single file**. Partitioning tiny outputs creates many small files, which
is a classic big-data anti-pattern (the "small files problem": each file costs metadata and
an open handle, and the overhead swamps the data).

In [ ]:
marts = {
    "overall_kpis":       kpis,
    "airline_metrics":    airline_metrics,
    "airport_metrics":    airport_metrics,
    "route_metrics":      route_metrics,
    "time_trends":        time_trends,
    "delay_distribution": delay_distribution,
    "delay_causes":       delay_causes,
    "airline_airport":    airline_airport,
}

print(f"{'MART':<22}{'ROWS':>10}")
print("-" * 32)
t0 = time.time()
counts = {}
for name, df in marts.items():
    df = df.coalesce(1).cache()
    counts[name] = df.count()
    df.write.mode("overwrite").parquet(str(PATHS["marts"] / f"{name}.parquet"))
    print(f"{name:<22}{counts[name]:>10,}")
print("-" * 32)
print(f"Written in {time.time()-t0:.1f}s")

In [ ]:
import subprocess
size = subprocess.run(["du","-sh",str(PATHS["marts"])], capture_output=True, text=True).stdout.split()[0]
print(f"All marts on disk : {size}")
print(f"Source dataset    : 201M curated / 565M raw CSV")
print()
print("This is the serving-layer argument in one line: the dashboard reads a few")
print("hundred KB instead of scanning 5.8 million rows on every interaction.")

---
## 9. Validation

Cross-check the aggregates against independently computed values. A mart that disagrees
with a direct query is worse than no mart at all.

In [ ]:
k = kpis.first()

# Independent recomputation, straight from the curated data.
direct_total     = flights.count()
direct_cancelled = flights.filter(CANCELLED).count()
direct_delayed   = flights.filter(COMPLETED & (F.col("is_delayed") == 1)).count()
direct_completed = flights.filter(COMPLETED).count()

checks = [
    ("total flights",     k["total_flights"], direct_total),
    ("cancellation rate", k["cancellation_rate"], round(100*direct_cancelled/direct_total, 2)),
    ("delay rate",        k["delay_rate"], round(100*direct_delayed/direct_completed, 2)),
    ("on-time pct",       k["on_time_pct"], round(100*(direct_completed-direct_delayed)/direct_completed, 2)),
]

print(f"{'METRIC':<20}{'MART':>12}{'DIRECT':>12}   OK")
print("-" * 50)
ok = True
for label, mart_val, direct_val in checks:
    match = abs(float(mart_val) - float(direct_val)) < 0.01
    ok &= match
    print(f"{label:<20}{mart_val:>12}{direct_val:>12}   {'yes' if match else 'NO'}")
assert ok, "a mart disagrees with a direct query"
print("\nAll KPIs reconcile.")

In [ ]:
# Airport departures must sum to the dataset; routes likewise.
a_sum = airport_metrics.agg(F.sum("total_flights")).first()[0]
r_sum = route_metrics.agg(F.sum("total_flights")).first()[0]
print(f"Airport totals : {a_sum:,}  (dataset {N:,})  {'OK' if a_sum == N else 'MISMATCH'}")
print(f"Route totals   : {r_sum:,}  (dataset {N:,})  {'OK' if r_sum == N else 'MISMATCH'}")
assert a_sum == N and r_sum == N

---
## 10. Summary

| Module | Output | Rows |
|---|---|---|
| 5 — Flight operations | `overall_kpis`, `delay_distribution`, `delay_causes` | small |
| 6 — Airline performance | `airline_metrics` | 14 |
| 7 — Airport intelligence | `airport_metrics` | 322 |
| 8 — Route intelligence | `route_metrics` | ~4,700 |
| — Time analysis | `time_trends` (4 grains) | ~50 |
| — Drill-down | `airline_airport` | ~1,500 |

Every ranking carries its sample size and a `meets_min_sample` flag, so the dashboard can
present fair comparisons without recomputing anything.

### Next
`06_ml_classification.ipynb` — delay prediction, using only features known before
departure. `airport_metrics` and `route_metrics` from this notebook supply the historical
delay-rate features.

In [ ]:
flights.unpersist()
spark.stop()
print("Notebook 05 complete. Marts ready for the dashboard and notebooks 06-08.")